# DTL: Feature Extraction and Generate Probability Layer using OVR method

# 1. Import satellite image from asset

In [1]:
!python -m pip install .. --quiet

import ee 

ee.Authenticate() 
ee.Initialize(project='epistem2')

# aoi definition

# import region from Hadi's assets
# region_name = "Sumatera"
# regions_fc = ee.FeatureCollection("users/hadicu06/IIASA/RESTORE/vector_datasets/classification_regions")
# aoi = regions_fc.filter(ee.Filter.eq('region_name', region_name)).geometry()

# try province export (aceh)

provinces = ee.FeatureCollection('projects/epistem2/assets/AOI_Sumatra_Provinces')
province_list = provinces.toList(provinces.size())
aoi = ee.Feature(province_list.get(9)).geometry()

# Load satellite image stack from asset

stacked_landsat = ee.Image('projects/epistem2/assets/stacked_landsat_2020_Sumatera')

# 2. Classification scheme 

In [2]:
from luma_ge.classification_scheme import LULC_Scheme_Manager

manager = LULC_Scheme_Manager()

scheme_name = "Epistem"
success, message = manager.load_default_scheme(scheme_name)
classification_df = manager.get_dataframe()

print(classification_df.to_string(index=False))

 ID          Land Cover Class Color Palette
  1    Primary Dryland Forest       #006400
  2  Secondary Dryland Forest       #228B22
  3   Primary Mangrove Forest       #4169E1
  4 Secondary Mangrove Forest       #87CEEB
  5      Primary Swamp Forest       #2E8B57
  6    Secondary Swamp Forest       #8FBC8F
  7         Plantation Forest       #32CD32
  8        Rubber Monoculture       #8B4513
  9      Oil palm Monoculture       #FF8C00
 10         Cacao Monoculture       #D2691E
 11       Coconut monoculture       #F4A460
 12         Other Monoculture       #DAA520
 13            Other Cropland       #FFFF00
 14       Coffee agroforestry       #6B8E23
 15       Rubber agroforestry       #9ACD32
 16         Mixed/home garden       #7CFC00
 17               Paddy field       #EEE8AA
 18          Grass or Savanna       #ADFF2F
 19                     Shrub       #90EE90
 20                Settlement       #FF0000
 21              Cleared Land       #D2B48C
 22               Mining area   

# 3. Load labelled training data from local

In [3]:
import geopandas as gpd
from shapely.geometry import shape
import geemap

TrainVectPath = "../data/modular_mapping_approach/sumatra_test/sumatra_td_DTL_result_v5_cleaned.shp"
TrainData = gpd.read_file(TrainVectPath)
if TrainData.crs is None:
    TrainData = TrainData.set_crs("EPSG:4326")
aoi_geojson = aoi.getInfo()
aoi_geom = shape(aoi_geojson)
aoi_gdf = gpd.GeoDataFrame(
    {"geometry": [aoi_geom]},
    crs="EPSG:4326"
)
aoi_gdf = aoi_gdf.to_crs(TrainData.crs)

    
TrainDataFinal = gpd.clip(
    TrainData,
    aoi_gdf
)

print(f"Total training points : {len(TrainData)}")
print(f"Points inside AOI     : {len(TrainDataFinal)}")
print(f"Points removed        : {len(TrainData) - len(TrainDataFinal)}")
print(TrainData.columns.tolist())

Total training points : 7078
Points inside AOI     : 722
Points removed        : 6356
['AoI', 'ID', 'LULC_24', 'agricultur', 'artificial', 'bareSoil_c', 'builtup_co', 'cacao_pres', 'coconut_pr', 'coffee_pre', 'mangrove_p', 'mining', 'oilpalm_pr', 'paddy_pres', 'rubber_pre', 'timber_ext', 'tree_cover', 'tree_heigh', 'waterbody_', '_qc_flag', 'longitude', 'latitude', 'label', 'class_name', 'geometry']


# 4. Feature extraction from satellite imageries

In [4]:
import geemap

# Settings

CLASS_PROPERTY = 'label'

N_TREES = 30
MIN_LEAF = 10
SEED = 0

band_names = stacked_landsat.bandNames()
labeled_roi = geemap.gdf_to_ee(TrainDataFinal)

In [5]:
# import geemap

# Map = geemap.Map()

# Map.centerObject(aoi, 7)
# Map.addLayer(aoi, {'color': 'red'}, 'AOI')
# Map.addLayer(stacked_landsat, {}, 'stacked_landsat_2020_Sumatera')
# Map.addLayer(labeled_roi, {}, 'labeled_roi')

# Map

In [6]:
from luma_ge.classification import FeatureExtraction

feature_extractor = FeatureExtraction()

stratified_train, stratified_test = feature_extractor.stratified_split(
                                                    labeled_roi, 
                                                    stacked_landsat, 
                                                    class_prop=CLASS_PROPERTY, 
                                                    train_ratio=0.7
                                                )

# remove unecessary properties (i.e. primitives) from the training and testing datasets

input_props = band_names.add(CLASS_PROPERTY)
stratified_train = stratified_train.select(input_props)
stratified_test  = stratified_test.select(input_props)

Stratified Random Split Training Pixel Size: 509
Stratified Random Split Testing Pixel Size: 213


# 5. Apply OVR classification

In [7]:
from luma_ge.classification import Generate_LULC

classifier = Generate_LULC()


probability_stack = classifier.soft_classification(
    training_data=stratified_train,
    class_property=CLASS_PROPERTY,
    image=stacked_landsat,
    include_final_map=False,
    ntrees=N_TREES,
    v_split=None,
    min_leaf=MIN_LEAF,
    seed=SEED,
    probability_scale=100
)


In [8]:
# sanity check

# 1. Check number and names of probability bands
# print("Bands:", probability_stack.bandNames().getInfo())
# print("Number of bands:", probability_stack.bandNames().size().getInfo())

# # 2. Check pixel type (should be Byte because of .byte())
# print("Image type:", probability_stack.bandTypes().getInfo())

# # 3. Check a single pixel only
# sample = probability_stack.sample(
#     region=aoi.centroid(),
#     scale=aoi.projection().nominalScale(),
#     numPixels=1,
#     geometries=False
# )

# print("Single-pixel probability values:")
# print(sample.first().getInfo())

In [10]:
# Export the probability stack to an asset

# Export to Earth Engine Asset
task = ee.batch.Export.image.toAsset(
    image=probability_stack,
    description='probability_stack_Sumsel_2020_td70',
    assetId='projects/epistem2/assets/probability_stack_Sumsel_2020_td70',
    region=aoi,
    scale=100,
    maxPixels=1e13
)

task.start()